# CEBRA paper figures — GPU run

Regenerates the CEBRA embedding (`combo_v`) on a GPU and builds the two
manuscript figures that live on the embedding:

* **Figure 3** — CEBRA trajectory globes (atlas + recovery / gray-zone / non-recovery)
* **Figure 2** — the digital twin rendered in CEBRA space (query + retrieved analogues)

**Runtime → Change runtime type → GPU (T4 is enough).**


## 1. Environment

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


In [ ]:
%pip -q install "cebra>=0.4.0" kaleido plotly scikit-learn tqdm
import cebra, plotly; print('cebra', cebra.__version__, '| plotly', plotly.__version__)


## 2. Mount Drive and locate the inputs

Needs four files from the team Drive. If the folder shows up under *Shared with
me*, right-click it → **Organise → Add shortcut to Drive** first, otherwise
`drive.mount` cannot see it.

| file | used by |
|---|---|
| `4_27_2026_train_300s_ppnet_plus.npz` | steps 01/02 |
| `4_27_2026_test_300s_ppnet_plus.npz`  | steps 01/02 |
| `twin_step4_handoff.npz`   | Figure 2 |
| `twin_matching_handoff.npz`| Figure 2 |


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, glob

def find(name, required=True):
    hits = glob.glob(f'/content/drive/**/{name}', recursive=True)
    if not hits:
        if required:
            raise FileNotFoundError(
                f'{name} not found under /content/drive. Add a shortcut to the '
                'team folder into My Drive and re-run.')
        print(f'  [optional] {name} not found — Figure 2 will be skipped')
        return None
    return sorted(hits, key=len)[0]

DATA_DIR, OUT_DIR, TWIN_DIR = '/content/data', '/content/out', '/content/twin'
for p in (DATA_DIR, OUT_DIR, TWIN_DIR): os.makedirs(p, exist_ok=True)

for src_name, dst in [
    ('4_27_2026_train_300s_ppnet_plus.npz', f'{DATA_DIR}/PPNet_data_train.npz'),
    ('4_27_2026_test_300s_ppnet_plus.npz',  f'{DATA_DIR}/PPNet_data_test.npz'),
]:
    src = find(src_name)
    if not os.path.exists(dst): os.symlink(src, dst)

HAVE_TWIN = True
for src_name, dst in [('twin_step4_handoff.npz',    f'{TWIN_DIR}/twin_step4_handoff.npz'),
                      ('twin_matching_handoff.npz', f'{TWIN_DIR}/twin_matching_handoff.npz')]:
    src = find(src_name, required=False)
    if src is None: HAVE_TWIN = False
    elif not os.path.exists(dst): os.symlink(src, dst)

os.environ['CEBRA_DATA_DIR'] = DATA_DIR
os.environ['CEBRA_OUT_DIR']  = OUT_DIR
print('\ndata:', os.listdir(DATA_DIR), '\ntwin:', os.listdir(TWIN_DIR))


## 3. Pipeline code

In [ ]:
%cd /content
![ -d Multimodal_Coma_Recovery ] || git clone -q https://github.com/ahmostafa147/Multimodal_Coma_Recovery.git
%cd /content/Multimodal_Coma_Recovery/cebra_pipeline
!ls


In [ ]:
# _constants.py: env-overridable paths + the verified twin class-order note
import re, pathlib
PALETTE = '''CLASS_COLORS = {
    0: '#c6362d',   # Seizure        - red
    1: '#fc7ebd',   # LPD            - pink
    2: '#d89009',   # GPD            - amber
    3: '#128763',   # LRDA           - dark teal
    4: '#36cf75',   # GRDA           - green
    5: '#872490',   # BurstSupp      - purple
    6: '#2d75d8',   # Continuous     - blue
    7: '#51bdf3',   # Discontinuous  - light blue
}'''
p = pathlib.Path('_constants.py'); s = p.read_text()
if 'CEBRA_DATA_DIR' not in s:
    s = re.sub(r"(?m)^DATA_DIR\s*=.*$",
               "import os as _os\nDATA_DIR = _os.environ.get('CEBRA_DATA_DIR', '/content/data')", s)
    s = re.sub(r"(?m)^OUT_DIR\s*=.*$",
               "OUT_DIR  = _os.environ.get('CEBRA_OUT_DIR', '/content/out')", s)
if 'TWIN_TO_OURS' not in s:
    s += ('\n\n# Twin handoff class axis order. Its meta JSON lists a DIFFERENT\n'
          "# class_names order, but that string is stale: the arrays are already in\n"
          '# CLASS_NAMES order. Verified on ICARE_0279 (L1 0.018 vs 0.728).\n'
          'TWIN_TO_OURS = [0, 1, 2, 3, 4, 5, 6, 7]\n')
# validated categorical palette (all-pairs: normal dE 20.1, CVD dE 8.0)
s = re.sub(r'CLASS_COLORS = \{.*?\n\}', PALETTE, s, flags=re.S)
p.write_text(s)
print(s[-560:])


In [ ]:
%%writefile cebra_figures.py
"""
cebra_figures.py — reusable, patient-agnostic Plotly builders for the
CEBRA figures in the EEG-Twin manuscript.

Everything keys off _constants.py, so the manuscript colour schema is the
single source of truth. Nothing here hardcodes a patient ID.

Typical use:
    import cebra_figures as cf
    run  = cf.load_run('combo_v')
    fig  = cf.fig_trajectory_globes(run, good_pid='ICARE_0279',
                                         poor_pid='ICARE_0231')
    cf.export(fig, 'Fig3_cebra_globes')
"""
import os
import re
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter

from _constants import (CLASS_NAMES, CLASS_COLORS, DISPLAY_ORDER, N_CLASSES,
                        OUTCOME_GOOD_RGB, OUTCOME_BAD_RGB, OUT_DIR)

# ── publication defaults ────────────────────────────────────────────
FONT      = 'Arial'
EXPORT_SCALE = 3          # ~600 dpi at the default pixel sizes
LABEL_RADIUS = 1.50       # state-label distance, in units of cloud radius
# Regions named in-scene on the atlas panel; the legend names all eight.
# Default = the recovery-axis anchors plus the two commonest periodic states.
ATLAS_LABEL_CLASSES = tuple(range(8))   # name every region

# One shared orientation for every globe in the paper, resolved jointly with
# the Fig-3 exemplars by optimal_camera() over every Fig-2 and Fig-3 trajectory:
#   5/8 regions forward, >=93% of the WORST trajectory exposed, both endpoints
#   of every trajectory on the visible face, regions well separated.
# All eight regions cannot face the viewer at once -- the class centroids span
# 171 deg -- so the three at the back stay named by their ring labels and
# leader lines, and read through the translucent cloud.
PAPER_CAMERA = dict(
    eye=dict(x=-0.42769188403952535, y=1.6529493873128962, z=1.0415555555555556),
    up=dict(x=0.1304524092482481, y=-0.5041742384815869, z=0.8536922783842193),
)
PANEL_LETTER_SIZE = 20


# ════════════════════════════════════════════════════════════════════
# Loading
# ════════════════════════════════════════════════════════════════════
def load_run(tag, out_dir=None, splits=('train', 'test')):
    """Load prep + embeddings for a run tag into a dict of split dicts."""
    out_dir = out_dir or OUT_DIR
    run = {'tag': tag}
    for s in splits:
        p = np.load(f'{out_dir}/cebra_prep_{tag}_{s}.npz', allow_pickle=True)
        e = np.load(f'{out_dir}/cebra_embeddings_{tag}_{s}.npz')['embedding']
        run[s] = dict(
            emb=e,
            y=p['predictions'].astype(int),
            cpc=p['cpc_scores'].astype(float),
            cpc_b=p['cpc_binary'].astype(int),
            pid=p['patient_ids'],
            times=p['times'].astype(float),
            acts=p['activations'],
            dur=p['bin_durations'].astype(float),
        )
    return run


def find_patient(run, pid):
    """Return (split_name, per-patient dict sorted by time). Raises if absent."""
    for s in ('train', 'test'):
        d = run.get(s)
        if d is None or pid not in d['pid']:
            continue
        m = d['pid'] == pid
        order = np.argsort(d['times'][m])
        t = d['times'][m][order]
        return s, dict(
            pid=pid, split=s,
            emb=d['emb'][m][order],
            y=d['y'][m][order],
            cpc=float(d['cpc'][m][0]),
            times=t,
            hours=(t - t[0]) / 3600.0,
        )
    raise KeyError(f'{pid} not found in {list(run.keys())}')


def list_patients(run, split='test', outcome=None, min_bins=0):
    """Patient IDs, optionally filtered by outcome ('good'/'poor') and length."""
    d = run[split]
    out = []
    for pid in np.unique(d['pid']):
        m = d['pid'] == pid
        cpc = float(d['cpc'][m][0])
        oc = 'good' if cpc <= 2 else 'poor' if cpc <= 5 else 'unknown'
        if outcome and oc != outcome:
            continue
        if int(m.sum()) < min_bins:
            continue
        out.append(pid)
    return out


# ════════════════════════════════════════════════════════════════════
# Geometry
# ════════════════════════════════════════════════════════════════════
def class_centroids(run, split='train', normalize=True):
    d = run[split]
    C = np.zeros((N_CLASSES, d['emb'].shape[1]))
    for c in range(N_CLASSES):
        m = d['y'] == c
        if m.any():
            C[c] = d['emb'][m].mean(axis=0)
            if normalize:
                C[c] /= np.linalg.norm(C[c]) + 1e-12
    return C


def small_circle(points, n_pts=160, percentile=68, cap=np.pi * 0.35,
                 label_radius=1.34):
    """Small circle on the sphere enclosing `percentile`% of a class cloud."""
    c = points.mean(axis=0)
    c_dir = c / (np.linalg.norm(c) + 1e-12)
    r = float(np.linalg.norm(points, axis=1).mean())
    pu = points / (np.linalg.norm(points, axis=1, keepdims=True) + 1e-12)
    ang = np.arccos(np.clip(pu @ c_dir, -1, 1))
    alpha = float(min(np.percentile(ang, percentile), cap))
    t = np.array([1.0, 0, 0]) if abs(c_dir[0]) < 0.9 else np.array([0, 1.0, 0])
    x = t - (t @ c_dir) * c_dir
    x /= np.linalg.norm(x) + 1e-12
    y = np.cross(c_dir, x)
    th = np.linspace(0, 2 * np.pi, n_pts)
    circ = ((np.cos(th)[:, None] * x + np.sin(th)[:, None] * y) * np.sin(alpha)
            + np.cos(alpha) * c_dir) * r
    return circ, c_dir * r * label_radius


def slerp(p0, p1, t):
    n0, n1 = np.linalg.norm(p0), np.linalg.norm(p1)
    if n0 < 1e-12 or n1 < 1e-12:
        return p0 * (1 - t) + p1 * t
    u0, u1 = p0 / n0, p1 / n1
    dot = float(np.clip(np.dot(u0, u1), -1.0, 1.0))
    om = np.arccos(dot)
    if om < 1e-6:
        return p0 * (1 - t) + p1 * t
    s = np.sin(om)
    return ((np.sin((1 - t) * om) / s) * u0
            + (np.sin(t * om) / s) * u1) * (n0 * (1 - t) + n1 * t)


def _sphere_normalize(P):
    return P / (np.linalg.norm(P, axis=-1, keepdims=True) + 1e-12)


def _catmull_rom(P, n_per_seg=12):
    """
    Centripetal Catmull-Rom through P. C1-continuous, so no kink at the knots
    (piecewise SLERP is only C0, which is what made the old path look jagged).
    """
    P = np.asarray(P, float)
    if len(P) < 3:
        return P.copy()
    ext = np.vstack([2 * P[0] - P[1], P, 2 * P[-1] - P[-2]])
    t = np.linspace(0, 1, n_per_seg, endpoint=False)[:, None]
    t2, t3 = t * t, t * t * t
    out = []
    for i in range(len(P) - 1):
        p0, p1, p2, p3 = ext[i], ext[i + 1], ext[i + 2], ext[i + 3]
        out.append(0.5 * ((2 * p1)
                          + (-p0 + p2) * t
                          + (2 * p0 - 5 * p1 + 4 * p2 - p3) * t2
                          + (-p0 + 3 * p1 - 3 * p2 + p3) * t3))
    out.append(P[-1][None, :])
    return np.vstack(out)


SMOOTH_SIGMA_BINS = 9      # 45 min at 5-min resolution


def smooth_trajectory(emb, y, window=24, slerp_n=10, method='gaussian',
                      sigma_bins=None, n_render=900, waypoints=True):
    """
    A smooth path along a patient's course through the embedding.

    method='gaussian' (default)
        Gaussian-smooth the raw per-segment sequence, project back onto the
        sphere, then resample with a centripetal Catmull-Rom spline.  The line
        follows every segment rather than a sparse polyline, and is C1, so it
        reads as one continuous arc.
    method='slerp'
        The original modal-state waypoints joined by piecewise SLERP. Kept so
        older figures reproduce; visibly kinked.

    Waypoint markers (state-coloured) are placed ON the smoothed path at window
    centres, so they annotate the curve instead of defining it.
    """
    emb = np.asarray(emb, float)
    N = len(emb)
    if N < 2:
        return None

    if method == 'slerp':
        wp, wt, ws = [], [], []
        for s0 in range(0, N, window):
            s1 = min(s0 + window, N)
            seg_y, seg_e = y[s0:s1], emb[s0:s1]
            cls, cnt = np.unique(seg_y, return_counts=True)
            win = int(cls[np.argmax(cnt)])
            last = np.where(seg_y == win)[0][-1]
            wp.append(seg_e[last]); wt.append((s0 + last) / max(N - 1, 1))
            ws.append(win)
        wp, wt = np.array(wp), np.array(wt)
        pts, ts = [], []
        for i in range(len(wp) - 1):
            for k in range(slerp_n):
                f = k / slerp_n
                pts.append(slerp(wp[i], wp[i + 1], f))
                ts.append(wt[i] + f * (wt[i + 1] - wt[i]))
        pts.append(wp[-1]); ts.append(wt[-1])
        return dict(waypoints=wp, waypoint_t=wt, waypoint_state=np.array(ws),
                    path=np.array(pts), path_t=np.array(ts))

    # ── gaussian: smooth the raw sequence, stay on the sphere ──────────
    from scipy.ndimage import gaussian_filter1d
    # 9 bins (45 min) chosen by sweep: it maximises the fraction of windows
    # whose smoothed point still sits nearest its own dominant state
    # (0.80/0.78 on two 8-state patients, vs 0.74/0.75 at sigma=12) while the
    # drawn line is already smooth (mean turn ~5 deg). Larger sigma buys a
    # little smoothness by erasing the state structure the figure is about.
    sigma = float(SMOOTH_SIGMA_BINS) if sigma_bins is None else float(sigma_bins)
    sigma = max(sigma, 0.5)
    sm = gaussian_filter1d(emb, sigma=sigma, axis=0, mode='nearest')
    sm = _sphere_normalize(sm)
    t_raw = np.linspace(0.0, 1.0, N)

    # thin before splining so the spline smooths rather than re-tracing noise
    n_knots = int(np.clip(N // max(int(window // 2), 1), 4, 400))
    idx = np.unique(np.linspace(0, N - 1, n_knots).astype(int))
    knots, knot_t = sm[idx], t_raw[idx]

    n_per_seg = max(int(np.ceil(n_render / max(len(knots) - 1, 1))), 2)
    path = _sphere_normalize(_catmull_rom(knots, n_per_seg))
    path_t = _catmull_rom(knot_t[:, None], n_per_seg)[:, 0]
    path_t = np.clip(path_t, 0.0, 1.0)

    wp = wt = ws = None
    if waypoints:
        wp, wt, ws = [], [], []
        for s0 in range(0, N, window):
            s1 = min(s0 + window, N)
            seg_y = y[s0:s1]
            cls, cnt = np.unique(seg_y, return_counts=True)
            ws.append(int(cls[np.argmax(cnt)]))
            mid = (s0 + s1 - 1) // 2          # window centre, on the smooth path
            wp.append(sm[mid]); wt.append(t_raw[mid])
        wp, wt, ws = np.array(wp), np.array(wt), np.array(ws)

    return dict(waypoints=wp, waypoint_t=wt, waypoint_state=ws,
                path=path, path_t=path_t, smoothed=sm)


def trajectory_arc(patient, centroids, edge=0.25):
    """
    Where a patient's trajectory starts and ends, in state terms.
      drift          cosine distance between the early and late trajectory mean
      to_continuous  gain in cosine similarity to the Continuous centroid
                     (positive = background recovers toward continuous)
      from_suppressed  loss of similarity to the Burst-Suppression centroid
    These summarise the recovery axis the manuscript argues from (S2.10).
    """
    e = patient['emb']
    n = len(e)
    k = max(int(n * edge), 1)
    a = e[:k].mean(axis=0)
    b = e[-k:].mean(axis=0)
    au = a / (np.linalg.norm(a) + 1e-12)
    bu = b / (np.linalg.norm(b) + 1e-12)
    cont = centroids[6] / (np.linalg.norm(centroids[6]) + 1e-12)
    bs = centroids[5] / (np.linalg.norm(centroids[5]) + 1e-12)
    return dict(
        drift=float(1.0 - np.dot(au, bu)),
        to_continuous=float(np.dot(bu, cont) - np.dot(au, cont)),
        from_suppressed=float(np.dot(au, bs) - np.dot(bu, bs)),
    )


def rank_by_arc(run, split='test', outcome='good', min_hours=40,
                centroids=None, top=8):
    """
    Rank patients by how clearly they show the recovery axis, not by how
    much they thrash.  Good-outcome patients are ranked by movement toward
    the Continuous region; poor-outcome patients by the absence of it.
    Returns [(pid, score, info), ...] best first.
    """
    C = centroids if centroids is not None else class_centroids(run)
    d = run[split]
    rows = []
    for pid in np.unique(d['pid']):
        m = d['pid'] == pid
        cpc = float(d['cpc'][m][0])
        oc = 'good' if cpc <= 2 else 'poor' if cpc <= 5 else 'unknown'
        if oc != outcome:
            continue
        order = np.argsort(d['times'][m])
        y = d['y'][m][order]
        hours = len(y) * 5 / 60.0
        if hours < min_hours:
            continue
        pat = dict(emb=d['emb'][m][order], y=y)
        arc = trajectory_arc(pat, C)
        n_states = len(np.unique(y))
        # legibility: reward a clear net displacement, mildly reward richness,
        # and do NOT reward raw transition count (that just makes scribble).
        base = arc['to_continuous'] if outcome == 'good' else -arc['to_continuous']
        score = base + 0.25 * arc['drift'] + 0.05 * n_states
        rows.append((pid, float(score),
                     dict(cpc=cpc, hours=hours, n_states=n_states,
                          transitions=int((y[1:] != y[:-1]).sum()), **arc)))
    rows.sort(key=lambda r: r[1], reverse=True)
    return rows[:top]


def prototype_landmarks(run, split='train', top_k=50):
    """Project each ProtoPNet prototype into the embedding (mean of top-k)."""
    d = run[split]
    acts = d['acts']
    n_proto = acts.shape[1]
    pos = np.zeros((n_proto, d['emb'].shape[1]))
    state = np.zeros(n_proto, dtype=int)
    for p in range(n_proto):
        top = np.argpartition(acts[:, p], -top_k)[-top_k:]
        pos[p] = d['emb'][top].mean(axis=0)
        state[p] = Counter(d['y'][top].tolist()).most_common(1)[0][0]
    return pos, state


def axis_range(*point_arrays, pad=1.10):
    pts = np.concatenate([np.atleast_2d(a) for a in point_arrays if a is not None
                          and len(a)], axis=0)
    half = float(np.abs(pts).max() * pad)
    return (-half, half)


def clinical_camera(centroids, elevation=0.40, distance=2.0):
    """
    Orient the camera so the burst-suppression -> continuous axis (the recovery
    axis the manuscript argues from, S2.10) lies exactly in the image plane,
    where motion along it is maximally visible.

    Build an orthonormal frame (a, w, eye) with a = the recovery axis and
    w = the part of world-up perpendicular to a.  Tilting the eye toward w
    keeps a . eye == 0, so the axis is never foreshortened.
    """
    bs, cont = centroids[5], centroids[6]
    a = cont - bs
    n = np.linalg.norm(a)
    if n < 1e-9:
        return dict(eye=dict(x=1.6, y=1.6, z=1.1), up=dict(x=0, y=0, z=1))
    a = a / n

    up = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(a, up)) > 0.95:
        up = np.array([0.0, 1.0, 0.0])
    w = up - np.dot(up, a) * a          # world-up, orthogonalised against a
    w /= np.linalg.norm(w) + 1e-12

    n_hat = np.cross(a, w)
    # face the camera toward the side the two anchor regions sit on, so the
    # recovery axis runs across the FRONT of the globe rather than behind it
    mid = 0.5 * (bs + cont)
    if np.dot(mid, n_hat) < 0:
        n_hat = -n_hat
    eye = n_hat + elevation * w            # both terms are perpendicular to a
    eye = eye / (np.linalg.norm(eye) + 1e-12) * distance
    return dict(eye=dict(x=float(eye[0]), y=float(eye[1]), z=float(eye[2])),
                up=dict(x=float(w[0]), y=float(w[1]), z=float(w[2])))


# ════════════════════════════════════════════════════════════════════
# Panel builders  (each appends traces to `fig` at row/col)
# ════════════════════════════════════════════════════════════════════
def _add_state_cloud(fig, emb, y, row, col, size=1.5, opacity=0.34,
                     legend=False, subsample=None, seed=0, legend_id='legend'):
    idx = np.arange(len(emb))
    if subsample and len(idx) > subsample:
        idx = np.random.RandomState(seed).choice(idx, subsample, replace=False)
    for c in DISPLAY_ORDER:
        m = y[idx] == c
        if not m.any():
            continue
        fig.add_trace(go.Scatter3d(
            x=emb[idx][m, 0], y=emb[idx][m, 1], z=emb[idx][m, 2],
            mode='markers',
            marker=dict(size=size, color=CLASS_COLORS[c], opacity=opacity),
            name=CLASS_NAMES[c], legendgroup=CLASS_NAMES[c],
            showlegend=False, legend=legend_id,
            hoverinfo='skip'), row=row, col=col)
        if legend:          # opaque proxy: swatches at cloud opacity read washed out
            fig.add_trace(go.Scatter3d(
                x=[None], y=[None], z=[None], mode='markers',
                marker=dict(size=7, color=CLASS_COLORS[c]),
                name=CLASS_NAMES[c], legendgroup=CLASS_NAMES[c],
                showlegend=True, legend=legend_id,
                hoverinfo='skip'), row=row, col=col)


NEUTRAL_CLOUD = '#b9bec6'


def _add_outcome_cloud(fig, emb, cpc_b, row, col, size=1.5, opacity=0.17,
                       legend=False, subsample=None, seed=0,
                       legend_id='legend2', balance=True, neutral=False):
    """
    Outcome cloud. Two corrections matter here:

    * the cohort is 441 poor vs 254 good, and poor patients record longer
      (228k vs 142k segments), so an unbalanced draw buries the good points;
      balance=True takes the same number from each outcome.
    * drawing the outcomes as two traces means the second is painted entirely
      on top of the first.  Both are merged into one shuffled trace with a
      per-point colour, so neither outcome occludes the other.

    Legend entries come from opaque zero-point proxies, since markers this
    small and transparent are invisible as swatches.
    """
    rs = np.random.RandomState(seed)
    if neutral:
        # one grey cloud: in the twin figure the cohort is context, and leaving
        # blue/orange unused there lets the analogues own those hues outright
        groups = [(0, NEUTRAL_CLOUD, 'Training cohort'),
                  (1, NEUTRAL_CLOUD, 'Training cohort')]
    else:
        groups = [(0, OUTCOME_GOOD_RGB, 'Good outcome (CPC 1-2)'),
                  (1, OUTCOME_BAD_RGB,  'Poor outcome (CPC 3-5)')]

    picks, colors = [], []
    for lab, color, _ in groups:
        idx = np.where(cpc_b == lab)[0]
        if not len(idx):
            continue
        if subsample:
            n = (subsample // 2 if balance
                 else int(subsample * len(idx) / len(cpc_b)))
            if len(idx) > n:
                idx = rs.choice(idx, n, replace=False)
        picks.append(idx)
        colors.append(np.full(len(idx), color, dtype=object))

    if picks:
        idx = np.concatenate(picks)
        col_arr = np.concatenate(colors)
        order = rs.permutation(len(idx))          # interleave the two outcomes
        idx, col_arr = idx[order], col_arr[order]
        e = emb[idx]
        fig.add_trace(go.Scatter3d(
            x=e[:, 0], y=e[:, 1], z=e[:, 2], mode='markers',
            marker=dict(size=size, color=list(col_arr), opacity=opacity),
            showlegend=False, legend=legend_id,
            hoverinfo='skip'), row=row, col=col)

    if legend:
        seen_names = set()
        for _, color, name in groups:
            if name in seen_names:
                continue
            seen_names.add(name)
            fig.add_trace(go.Scatter3d(
                x=[None], y=[None], z=[None], mode='markers',
                marker=dict(size=7, color=color),
                name=name, legendgroup=name, showlegend=True,
                legend=legend_id, hoverinfo='skip'), row=row, col=col)


def _fibonacci_directions(n):
    """n roughly uniform unit vectors on the sphere."""
    i = np.arange(n) + 0.5
    phi = np.arccos(1 - 2 * i / n)
    theta = np.pi * (1 + 5 ** 0.5) * i
    return np.stack([np.cos(theta) * np.sin(phi),
                     np.sin(theta) * np.sin(phi),
                     np.cos(phi)], axis=1)


def score_camera(eye, centroids, paths, min_sep_target=0.42):
    """
    How well one viewing direction serves the figure.

      states     how many of the 8 region centroids face the viewer
      sep        smallest 2-D gap between any two region centroids
                 (regions that project on top of each other are unreadable)
      front      fraction of trajectory points not hidden behind the globe
      spread     2-D extent of the trajectories (a foreshortened path is
                 visible but useless -- start, middle and end must be legible)
    """
    eye = eye / (np.linalg.norm(eye) + 1e-12)
    up = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(eye, up)) > 0.95:
        up = np.array([0.0, 1.0, 0.0])
    up = up - np.dot(up, eye) * eye
    up /= np.linalg.norm(up) + 1e-12
    right = np.cross(up, eye)

    C = centroids / (np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-12)
    depth = C @ eye
    # graded, not binary: the cloud is translucent, so a region just past the
    # rim still reads. All eight can never face the viewer at once -- the
    # centroids span 171 deg -- so reward getting as many forward as possible.
    states = float(np.mean(np.clip((depth + 0.35) / 0.70, 0.0, 1.0)))
    P2 = np.stack([C @ right, C @ up], axis=1)
    d = np.linalg.norm(P2[:, None, :] - P2[None, :, :], axis=2)
    d[np.arange(len(d)), np.arange(len(d))] = np.inf
    sep = float(min(d.min() / min_sep_target, 1.0))

    fronts, spreads, ends = [], [], []
    for path in paths:
        u = path / (np.linalg.norm(path, axis=1, keepdims=True) + 1e-12)
        dp = u @ eye
        fronts.append(float(np.mean(dp > -0.10)))
        # start and end markers must both be on the visible face; a path whose
        # endpoint is behind the globe fails "fully visible" no matter how much
        # of its middle shows
        ends.append(float(min(dp[0], dp[-1]) > -0.05))
        q = np.stack([u @ right, u @ up], axis=1)
        step = np.linalg.norm(np.diff(q, axis=0), axis=1).sum()
        box = (q[:, 0].ptp() * q[:, 1].ptp()) ** 0.5
        spreads.append(0.5 * min(step / 2.5, 1.0) + 0.5 * min(box / 1.1, 1.0))
    # worst case, not average: averaging lets one hidden trajectory hide behind
    # several well-exposed ones
    front = float(np.min(fronts)) if fronts else 1.0
    spread = float(np.min(spreads)) if spreads else 1.0
    endpoints = float(np.min(ends)) if ends else 1.0

    total = (2.6 * states + 1.2 * sep + 1.6 * front + 1.0 * spread
             + 1.5 * endpoints)
    return total, dict(states=states, sep=sep, front=front, spread=spread,
                       endpoints=endpoints)


def optimal_camera(run, paths=(), split='train', n_candidates=20000,
                   distance=2.0, front_min=0.90, sep_min=0.85, verbose=False):
    """
    One shared viewing direction for every globe in the paper.

    Constrained, not a plain maximum: the class centroids span 171 deg, so no
    view can put all eight regions in front. Trajectory exposure is the binding
    requirement (a path hidden behind the globe is unreadable), so `front` and
    `sep` are hard floors and the number of forward-facing regions is maximised
    subject to them. Deterministic.
    """
    C = class_centroids(run, split)
    U = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-12)
    paths = [np.asarray(p, float) for p in paths if p is not None and len(p)]
    best, best_key, best_parts = None, None, None
    for eye in _fibonacci_directions(n_candidates):
        sc, parts = score_camera(eye, C, paths)
        if paths and (parts['front'] < front_min or parts['sep'] < sep_min
                      or parts['endpoints'] < 1.0):
            continue
        key = (int(((U @ eye) > 0).sum()), parts['states'], sc)
        if best_key is None or key > best_key:
            best, best_key, best_parts = eye, key, parts
    if best is None:                      # floors unreachable -> plain maximum
        for eye in _fibonacci_directions(n_candidates):
            sc, parts = score_camera(eye, C, paths)
            if best_key is None or sc > best_key[-1]:
                best, best_key, best_parts = eye, (0, parts['states'], sc), parts
    if verbose:
        print(f'  camera: {best_key[0]}/8 regions forward  ' +
              '  '.join(f'{k}={v:.2f}' for k, v in best_parts.items()))
    up = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(best, up)) > 0.95:
        up = np.array([0.0, 1.0, 0.0])
    up = up - np.dot(up, best) * best
    up /= np.linalg.norm(up) + 1e-12
    e = best * distance
    return dict(eye=dict(x=float(e[0]), y=float(e[1]), z=float(e[2])),
                up=dict(x=float(up[0]), y=float(up[1]), z=float(up[2])))


def camera_frame(camera):
    """Orthonormal (right, up, eye) image-plane basis for a plotly camera."""
    e = camera.get('eye', {})
    eye = np.array([e.get('x', 0), e.get('y', 0), e.get('z', 0)], float)
    n = np.linalg.norm(eye)
    if n < 1e-9:
        return None
    eye = eye / n
    uu = camera.get('up', {'x': 0, 'y': 0, 'z': 1})
    up = np.array([uu.get('x', 0), uu.get('y', 0), uu.get('z', 1)], float)
    up = up - np.dot(up, eye) * eye
    up /= np.linalg.norm(up) + 1e-12
    return np.cross(up, eye), up, eye


def label_ring(run, camera, split='train', min_pts=20, ring=1.34,
               front=0.85, min_deg=30.0):
    """
    Lay the state names out on a ring around the globe.

    Each label keeps the compass bearing of its own cluster (so it still points
    at the right region), is pushed clear of the point cloud, and is placed at
    a front depth so it is never occluded.  Overlaps are resolved by sliding
    labels ALONG the ring, which cannot drag a label off its cluster the way
    free 2-D repulsion does.

    Returns {class_index: (label_xyz, anchor_xyz)} for drawing leader lines.
    """
    frame = camera_frame(camera)
    if frame is None:
        return {}
    right, up, eye = frame
    d = run[split]
    R = float(np.linalg.norm(d['emb'], axis=1).mean())

    items = []
    for c in DISPLAY_ORDER:
        m = d['y'] == c
        if m.sum() < min_pts:
            continue
        cen = d['emb'][m].mean(axis=0)
        cdir = cen / (np.linalg.norm(cen) + 1e-12)
        anchor = cdir * R
        th = float(np.arctan2(anchor @ up, anchor @ right))
        items.append([c, th, anchor])
    if not items:
        return {}

    # slide along the ring until every neighbour is >= min_deg apart
    items.sort(key=lambda it: it[1])
    step = np.deg2rad(min_deg)
    n = len(items)
    if n * step > 2 * np.pi:                      # cannot fit -> spread evenly
        base = items[0][1]
        for i, it in enumerate(items):
            it[1] = base + i * (2 * np.pi / n)
    else:
        for _ in range(400):
            moved = False
            for i in range(n):
                j = (i + 1) % n
                gap = items[j][1] - items[i][1]
                if j == 0:
                    gap += 2 * np.pi
                if gap < step:
                    push = (step - gap) / 2.0
                    items[i][1] -= push
                    items[j][1] += push
                    moved = True
            if not moved:
                break

    out = {}
    for c, th, anchor in items:
        pos = (ring * R) * (np.cos(th) * right + np.sin(th) * up) \
              + (front * R) * eye
        out[c] = (pos, anchor)
    return out


def _add_boundaries(fig, run, row, col, split='train', labels=True,
                    legend=False, width=5, min_pts=20, camera=None,
                    label_size=13, legend_id='legend', label_classes=None,
                    leaders=True, ring=1.34, min_deg=30.0):
    """State boundary circles, plus optional ring-laid state names."""
    d = run[split]
    for c in DISPLAY_ORDER:
        m = d['y'] == c
        if m.sum() < min_pts:
            continue
        circ, _ = small_circle(d['emb'][m])
        fig.add_trace(go.Scatter3d(
            x=circ[:, 0], y=circ[:, 1], z=circ[:, 2], mode='lines',
            line=dict(color=CLASS_COLORS[c], width=width),
            name=CLASS_NAMES[c], legendgroup=CLASS_NAMES[c],
            showlegend=legend, legend=legend_id,
            hoverinfo='skip'), row=row, col=col)

    if not labels or camera is None:
        return
    want = DISPLAY_ORDER if label_classes is None else list(label_classes)
    frame = camera_frame(camera)
    eye = frame[2] if frame is not None else None
    ring_pos = label_ring(run, camera, split, min_pts, ring, min_deg=min_deg)
    right_v = frame[0] if frame is not None else None
    span = max((abs(float(p @ right_v)) for p, _ in ring_pos.values()),
               default=1.0) if right_v is not None else 1.0
    for c, (pos, anchor) in ring_pos.items():
        if c not in want:
            continue
        # text is centred on its anchor and overflows it; near the frame edge
        # anchor it so the word grows inward instead of off the panel
        tpos = 'middle center'
        if right_v is not None and span > 1e-9:
            xr = float(pos @ right_v) / span
            if xr > 0.45:
                tpos = 'middle left'
            elif xr < -0.45:
                tpos = 'middle right'
        if leaders:
            # Start just outside the cloud, never at the centroid: a leader from
            # the centroid crosses the sphere and hides the points under it.
            # For a region on the FAR side, step out along the silhouette in the
            # same bearing instead, or the leader tunnels through the globe.
            outer = anchor * 1.07
            if eye is not None and float(anchor @ eye) < 0.0:
                rim = anchor - float(anchor @ eye) * eye
                n = np.linalg.norm(rim)
                if n > 1e-9:
                    outer = rim / n * np.linalg.norm(anchor) * 1.07
            fig.add_trace(go.Scatter3d(
                x=[outer[0], pos[0]], y=[outer[1], pos[1]],
                z=[outer[2], pos[2]], mode='lines',
                line=dict(color=CLASS_COLORS[c], width=1.6),
                opacity=0.6, showlegend=False,
                hoverinfo='skip'), row=row, col=col)
        fig.add_trace(go.Scatter3d(
            x=[pos[0]], y=[pos[1]], z=[pos[2]], mode='text',
            text=[f'<b>{CLASS_NAMES[c]}</b>'], textposition=tpos,
            textfont=dict(size=label_size, color=CLASS_COLORS[c], family=FONT),
            showlegend=False, hoverinfo='skip'), row=row, col=col)


def _add_trajectory(fig, traj, row, col, name, halo=True,
                    line_width=5, marker_size=3.2, legend=True,
                    colorbar=None, show_waypoints=True, legend_id='legend2'):
    p, t = traj['path'], traj['path_t']
    if halo:
        fig.add_trace(go.Scatter3d(
            x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='lines',
            line=dict(color='rgba(0,0,0,0.55)', width=line_width + 3),
            showlegend=False, hoverinfo='skip'), row=row, col=col)
    fig.add_trace(go.Scatter3d(
        x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='lines',
        line=dict(color=t, colorscale='Plasma', width=line_width,
                  cmin=0, cmax=1,
                  showscale=colorbar is not None,
                  colorbar=colorbar or None),
        name=name, showlegend=legend, legend=legend_id,
        hoverinfo='skip'), row=row, col=col)
    wp, wst = traj['waypoints'], traj['waypoint_state']
    if show_waypoints:
        fig.add_trace(go.Scatter3d(
            x=wp[:, 0], y=wp[:, 1], z=wp[:, 2], mode='markers',
            marker=dict(size=marker_size,
                        color=[CLASS_COLORS[s] for s in wst],
                        line=dict(color='black', width=0.8)),
            text=[CLASS_NAMES[s] for s in wst],
            hovertemplate='%{text}<extra></extra>',
            showlegend=False), row=row, col=col)
    for pt, sym, col_, lbl in [(wp[0], 'diamond', '#00C853', 'Start'),
                               (wp[-1], 'diamond', '#D50000', 'End')]:
        fig.add_trace(go.Scatter3d(
            x=[pt[0]], y=[pt[1]], z=[pt[2]], mode='markers',
            marker=dict(size=6, color=col_, symbol=sym,
                        line=dict(color='black', width=1.2)),
            name=lbl, legendgroup=lbl, showlegend=legend,
            legend=legend_id, hoverinfo='skip'), row=row, col=col)


def _scene(rng, camera, show_axes=False):
    ax = dict(range=rng, showbackground=False, showticklabels=False,
              title='', showgrid=show_axes, zeroline=False,
              showline=False, visible=show_axes)
    return dict(bgcolor='white', aspectmode='cube',
                xaxis=ax, yaxis=ax, zaxis=ax, camera=camera)


# ════════════════════════════════════════════════════════════════════
# Figure: CEBRA trajectory globes
# ════════════════════════════════════════════════════════════════════
def fig_trajectory_globes(run, patients=None, good_pid=None, poor_pid=None,
                          split='train', atlas=True, atlas_subsample=70000,
                          cloud_subsample=55000, window=24, slerp_n=10,
                          camera=None, show_prototypes=False,
                          panel_width=470, height=575,
                          label_ring_r=1.20, pad=1.16, label_size=11.5):
    """
    A row of CEBRA globes on one shared camera and one shared axis range.

    patients : list of (pid, caption) — any length, any patients, any split.
               e.g. [('ICARE_0142', 'Recovery'),
                     ('ICARE_0279', 'Gray zone'),
                     ('ICARE_0277', 'Non-recovery')]
               A bare 'PID' string is accepted and captioned automatically.
    atlas    : prepend the state-atlas panel (manifold coloured by ACNS state).

    good_pid/poor_pid are kept as a two-panel shorthand for older callers.
    """
    if patients is None:
        if good_pid is None or poor_pid is None:
            raise ValueError('pass patients=[...] or both good_pid and poor_pid')
        patients = [(good_pid, 'Good outcome'), (poor_pid, 'Poor outcome')]
    if not patients and not atlas:
        raise ValueError('nothing to draw: pass patients=[...] or atlas=True')

    norm = []
    for item in patients:
        pid, cap = (item, None) if isinstance(item, str) else item
        _, pat = find_patient(run, pid)
        if cap is None:
            cap = 'Good outcome' if pat['cpc'] <= 2 else 'Poor outcome'
        norm.append((pid, cap, pat, smooth_trajectory(pat['emb'], pat['y'],
                                                      window, slerp_n)))

    d = run[split]
    C = class_centroids(run, split)
    camera = camera or PAPER_CAMERA

    label_pts = [p[None, :] for p, _ in
                 label_ring(run, camera, split, ring=label_ring_r).values()]
    rng = axis_range(d['emb'], *[t['path'] for _, _, _, t in norm],
                     *label_pts, pad=pad)

    letters = 'abcdefghij'
    titles, k = [], 0
    if atlas:
        titles.append('<b>a</b>   CEBRA state atlas'); k = 1
    for i, (pid, cap, pat, _) in enumerate(norm):
        cpc_txt = '' if 'CPC' in cap else f"CPC {pat['cpc']:.0f}, "
        titles.append(f"<b>{letters[i + k]}</b>   {cap} — {pid} "
                      f"({cpc_txt}{pat['hours'][-1]:.0f} h)")

    ncols = len(titles)
    fig = make_subplots(rows=1, cols=ncols,
                        specs=[[{'type': 'scatter3d'}] * ncols],
                        subplot_titles=titles, horizontal_spacing=0.008)

    col = 1
    if atlas:
        _add_state_cloud(fig, d['emb'], d['y'], 1, 1, opacity=0.35,
                         legend=True, subsample=atlas_subsample,
                         legend_id='legend')
        _add_boundaries(fig, run, 1, 1, split=split, labels=True,
                        legend=False, camera=camera, width=5,
                        label_size=label_size, ring=label_ring_r,
                        label_classes=ATLAS_LABEL_CLASSES)
        if show_prototypes:
            pos, st = prototype_landmarks(run, split)
            for c in DISPLAY_ORDER:
                m = st == c
                if m.any():
                    fig.add_trace(go.Scatter3d(
                        x=pos[m, 0], y=pos[m, 1], z=pos[m, 2], mode='markers',
                        marker=dict(size=5, color=CLASS_COLORS[c],
                                    symbol='cross',
                                    line=dict(color='white', width=1)),
                        showlegend=False, hoverinfo='skip'), row=1, col=1)
        col = 2

    cbar = dict(title=dict(text='Recording<br>progress', font=dict(size=11)),
                x=1.005, len=0.55, thickness=12,
                tickvals=[0, 0.5, 1], ticktext=['start', 'mid', 'end'],
                tickfont=dict(size=10))
    for i, (pid, cap, pat, traj) in enumerate(norm):
        first, last = (i == 0), (i == len(norm) - 1)
        _add_outcome_cloud(fig, d['emb'], d['cpc_b'], 1, col,
                           legend=first, subsample=cloud_subsample,
                           legend_id='legend2')
        _add_boundaries(fig, run, 1, col, split=split, labels=False,
                        legend=False, width=2.0)
        _add_trajectory(fig, traj, 1, col, name='Patient trajectory',
                        legend=first, legend_id='legend2',
                        colorbar=cbar if last else None)
        col += 1

    sc = _scene(rng, camera)
    layout = dict(paper_bgcolor='white',
                  font=dict(color='black', family=FONT, size=12),
                  legend=dict(title=dict(text='<b>EEG state</b>  ',
                                         font=dict(size=10), side='left'),
                              orientation='h', font=dict(size=10),
                              itemsizing='constant', x=0.5, y=-0.02,
                              xanchor='center', yanchor='top'),
                  legend2=dict(title=dict(text='<b>Cohort</b>  ',
                                          font=dict(size=10), side='left'),
                               orientation='h', font=dict(size=10),
                               itemsizing='constant', x=0.5, y=-0.10,
                               xanchor='center', yanchor='top'),
                  margin=dict(l=0, r=95, t=40, b=88),
                  width=panel_width * ncols, height=height)
    for j in range(ncols):
        layout['scene' if j == 0 else f'scene{j + 1}'] = sc
    fig.update_layout(**layout)
    for a in fig.layout.annotations:
        a.font.size = 12.5
        a.font.family = FONT
    return fig


# ════════════════════════════════════════════════════════════════════
# Export
# ════════════════════════════════════════════════════════════════════
_CAMERA_READOUT = """
var gd = document.getElementById('{plot_id}');
var NL = String.fromCharCode(10);
var scenes = Object.keys(gd.layout).filter(function (k) {
  return k.indexOf('scene') === 0;
});
var box = document.createElement('div');
box.style.cssText = 'position:fixed;bottom:10px;left:10px;max-width:46em;' +
  'background:rgba(255,255,255,.96);border:1px solid #bbb;border-radius:5px;' +
  'padding:7px 10px;font:11px/1.5 ui-monospace,Menlo,monospace;z-index:9999;' +
  'white-space:pre-wrap;color:#222;box-shadow:0 1px 4px rgba(0,0,0,.15)';
box.textContent = (scenes.length > 1)
  ? 'Rotate any globe - all ' + scenes.length + ' follow. Camera appears here.'
  : 'Rotate the globe - its camera appears here, ready to paste.';
document.body.appendChild(box);
function f(v) { return (Math.round(v * 1000) / 1000); }
var syncing = false;
gd.on('plotly_relayout', function (e) {
  if (syncing) return;
  var key = null;
  for (var k in e) { if (k.indexOf('.camera') !== -1) { key = k; break; } }
  if (!key) return;
  var src = key.split('.')[0];
  var cam = gd.layout[src] && gd.layout[src].camera;
  if (!cam || !cam.eye) return;
  var u = cam.up || {x: 0, y: 0, z: 1};
  box.textContent =
    'camera (' + src + ')  -  paste as camera=...' + NL +
    'dict(eye=dict(x=' + f(cam.eye.x) + ', y=' + f(cam.eye.y) +
    ', z=' + f(cam.eye.z) + '),' + NL +
    '     up=dict(x=' + f(u.x) + ', y=' + f(u.y) + ', z=' + f(u.z) + '))';
  if (scenes.length > 1) {
    var upd = {};
    scenes.forEach(function (s) { if (s !== src) { upd[s + '.camera'] = cam; } });
    syncing = true;
    Plotly.relayout(gd, upd).then(function () { syncing = false; },
                                 function () { syncing = false; });
  }
});
"""


def export(fig, stem, out_dir=None, formats=('html', 'png', 'pdf'),
           scale=EXPORT_SCALE, camera_readout=True, selfcontained=False):
    """
    Write interactive HTML plus publication-resolution static files.

    The HTML carries a small overlay that prints the live camera as a Python
    dict: rotate a globe to taste, copy the dict, pass it back as `camera=` and
    the static export will match exactly what you framed.
    """
    out_dir = out_dir or os.path.join(OUT_DIR, 'figures')
    os.makedirs(out_dir, exist_ok=True)
    written = []
    for f in formats:
        path = os.path.join(out_dir, f'{stem}.{f}')
        if f == 'html':
            fig.write_html(path,
                           include_plotlyjs=True if selfcontained else 'cdn',
                           post_script=_CAMERA_READOUT if camera_readout else None)
        else:
            fig.write_image(path, scale=scale)
        written.append(path)
        print(f'  wrote {path}')
    return written


def export_single_globes(run, panels, stem_prefix='Fig3_globe',
                         out_dir=None, atlas=True, **kw):
    """
    One standalone HTML per globe — easier to orient and screenshot than the
    combined row, where every scene rotates independently anyway.
    """
    written = []
    jobs = ([('a_atlas', [], True)] if atlas else [])
    letters = 'bcdefgh' if atlas else 'abcdefg'
    for i, (pid, caption) in enumerate(panels):
        slug = re.sub(r'[^a-z0-9]+', '', caption.lower())
        tag = f'{letters[i]}_{slug}_{pid}'
        jobs.append((tag, [(pid, caption)], False))
    for tag, pats, want_atlas in jobs:
        fig = fig_trajectory_globes(run, patients=pats, atlas=want_atlas,
                                    panel_width=900, height=900, **kw)
        fig.update_layout(margin=dict(l=0, r=0, t=40, b=90))
        written += export(fig, f'{stem_prefix}_{tag}', out_dir=out_dir,
                          formats=('html',))
    return written


# ════════════════════════════════════════════════════════════════════
# Figure: patient trajectory with ProtoPNet prototypes
# ════════════════════════════════════════════════════════════════════
def fig_patient_prototypes(run, pid, split='train', top_k=50,
                           cloud_subsample=45000, window=24, slerp_n=10,
                           camera=None, n_highlight=5, width=1480, height=720,
                           panel_b=True):
    """
    One patient against the prototype landmarks.

      (a) the CEBRA manifold with all ProtoPNet prototypes projected into it
          (each placed at the mean embedding of its top-k activating segments,
          coloured by the phenotype it codes for), the named state regions, and
          the patient's trajectory. Prototypes the patient activates most
          strongly are ringed.
      (b) the patient's prototype activations over time, prototypes grouped by
          phenotype -- the evidence trail behind the trajectory in (a).

    panel_b=False gives the globe alone, square and full-size -- easier to
    rotate and screenshot than a globe squeezed into half a figure.
    """
    d = run[split]
    camera = camera or PAPER_CAMERA
    pos, pstate = prototype_landmarks(run, split, top_k)

    _, p = find_patient(run, pid)
    traj = smooth_trajectory(p['emb'], p['y'], window, slerp_n)

    src = run['test'] if pid in run['test']['pid'] else run['train']
    m = src['pid'] == pid
    order = np.argsort(src['times'][m])
    acts = src['acts'][m][order]                     # (T, 45)
    hours = (src['times'][m][order] - src['times'][m][order][0]) / 3600.0

    # prototypes this patient leans on most
    mean_act = acts.mean(axis=0)
    top_idx = np.argsort(-mean_act)[:n_highlight]

    label_pts = [q[None, :] for q, _ in label_ring(run, camera, split).values()]
    rng = axis_range(d['emb'], traj['path'], pos, *label_pts, pad=1.04)

    if panel_b:
        fig = make_subplots(
            rows=1, cols=2, column_widths=[0.5, 0.5],
            specs=[[{'type': 'scatter3d'}, {'type': 'xy'}]],
            subplot_titles=(
                f'<b>a</b>   {pid} and the {len(pos)} ProtoPNet prototypes '
                f'in CEBRA space',
                f'<b>b</b>   {pid} prototype activations over time'),
            horizontal_spacing=0.08)
    else:
        fig = make_subplots(
            rows=1, cols=1, specs=[[{'type': 'scatter3d'}]],
            subplot_titles=(f'{pid} and the {len(pos)} ProtoPNet prototypes '
                            f'in CEBRA space',))

    _add_outcome_cloud(fig, d['emb'], d['cpc_b'], 1, 1, opacity=0.12, size=1.4,
                       legend=True, subsample=cloud_subsample,
                       legend_id='legend')
    _add_boundaries(fig, run, 1, 1, split=split, labels=True, legend=False,
                    camera=camera, width=2.5, label_size=11)

    for c in DISPLAY_ORDER:
        sel = pstate == c
        if not sel.any():
            continue
        fig.add_trace(go.Scatter3d(
            x=pos[sel, 0], y=pos[sel, 1], z=pos[sel, 2], mode='markers',
            marker=dict(size=6, color=CLASS_COLORS[c], symbol='diamond',
                        line=dict(color='rgba(20,20,25,0.9)', width=1.4)),
            name=f'Prototype — {CLASS_NAMES[c]}', legendgroup='proto',
            showlegend=False, legend='legend',
            hovertemplate=f'{CLASS_NAMES[c]} prototype<extra></extra>'),
            row=1, col=1)
    fig.add_trace(go.Scatter3d(                       # one legend entry
        x=[None], y=[None], z=[None], mode='markers',
        marker=dict(size=8, color='#666', symbol='diamond',
                    line=dict(color='black', width=1.4)),
        name=f'ProtoPNet prototype (n={len(pos)})', legend='legend',
        showlegend=True, hoverinfo='skip'), row=1, col=1)
    fig.add_trace(go.Scatter3d(
        x=pos[top_idx, 0], y=pos[top_idx, 1], z=pos[top_idx, 2],
        mode='markers',
        marker=dict(size=13, color='rgba(0,0,0,0)', symbol='circle',
                    line=dict(color='#111', width=2.2)),
        name=f'Top-{n_highlight} for this patient', legend='legend',
        showlegend=True,
        text=[f'prototype {i} ({CLASS_NAMES[pstate[i]]})' for i in top_idx],
        hovertemplate='%{text}<extra></extra>'), row=1, col=1)

    _add_trajectory(fig, traj, 1, 1, name=f'{pid} trajectory',
                    legend=True, legend_id='legend', line_width=6,
                    marker_size=3.4)

    # ── (b) activation heatmap, prototypes grouped by phenotype ─────
    if not panel_b:
        fig.update_layout(
            scene=_scene(rng, camera),
            paper_bgcolor='white', plot_bgcolor='white',
            font=dict(color='black', family=FONT, size=12),
            legend=dict(title=dict(text='<b>CEBRA space</b>',
                                   font=dict(size=10)),
                        font=dict(size=10), itemsizing='constant',
                        x=0.005, y=0.99, xanchor='left', yanchor='top',
                        bgcolor='rgba(255,255,255,0.88)',
                        bordercolor='lightgray', borderwidth=1),
            margin=dict(l=0, r=0, t=44, b=0),
            width=950, height=950)
        for a in fig.layout.annotations[:1]:
            a.font.size = 13
            a.font.family = FONT
        return fig

    proto_order = [i for c in DISPLAY_ORDER
                   for i in np.where(pstate == c)[0]]
    fig.add_trace(go.Heatmap(
        z=acts[:, proto_order].T, x=hours,
        y=list(range(len(proto_order))),
        colorscale='Magma', zmin=0.0, zmax=float(acts.max()),
        colorbar=dict(title=dict(text='Prototype<br>activation',
                                 font=dict(size=10)),
                      x=1.005, len=0.62, thickness=12,
                      tickfont=dict(size=9)),
        hovertemplate='%{x:.1f} h · row %{y}: %{z:.3f}<extra></extra>'),
        row=1, col=2)
    # phenotype key down the left edge of the heatmap
    for row_i, pi in enumerate(proto_order):
        fig.add_trace(go.Scatter(
            x=[-1.8], y=[row_i], mode='markers',
            marker=dict(color=CLASS_COLORS[pstate[pi]], size=7,
                        symbol='square'),
            showlegend=False,
            hovertemplate=f'prototype {pi} — '
                          f'{CLASS_NAMES[pstate[pi]]}<extra></extra>'),
            row=1, col=2)

    fig.update_xaxes(title_text='Hours from recording start', row=1, col=2,
                     range=[-3.0, float(hours.max())])
    fig.update_yaxes(title_text='ProtoPNet prototype (grouped by phenotype)',
                     row=1, col=2, showticklabels=False,
                     autorange='reversed')

    fig.update_layout(
        scene=_scene(rng, camera),
        paper_bgcolor='white', plot_bgcolor='white',
        font=dict(color='black', family=FONT, size=12),
        legend=dict(title=dict(text='<b>CEBRA space</b>', font=dict(size=10)),
                    font=dict(size=9.5), itemsizing='constant',
                    x=0.005, y=0.30, xanchor='left', yanchor='top',
                    bgcolor='rgba(255,255,255,0.88)',
                    bordercolor='lightgray', borderwidth=1),
        margin=dict(l=0, r=20, t=52, b=52),
        width=width, height=height)
    for a in fig.layout.annotations[:2]:
        a.font.size = 12.5
        a.font.family = FONT
    fig.update_xaxes(showgrid=False, zeroline=False)
    fig.update_yaxes(showgrid=False, zeroline=False)
    return fig


In [ ]:
%%writefile twin_figures.py
"""
twin_figures.py — the digital twin rendered in CEBRA space (Figure 2).

Depends on Keaton's handoff files:
    twin/twin_step4_handoff.npz      per-block outcome, forecast, hidden state
    twin/twin_matching_handoff.npz   per-hour matching streams + roll_traj

Retrieval reproduces the deployed rule (manuscript S2.8):
    combined = alpha * trajectory + (1 - alpha) * feature
where trajectory is cosine similarity between transformer hidden states at the
query hour, and feature is the mean of per-stream standardised cosines over
CEBRA, ProtoPNet, prototype activations, label frequencies, qEEG and clinical.

Train patients with no EEG yet at the query hour carry NaN in the matching
streams and are excluded from the pool -- the same constraint a bedside system
would face.
"""
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import cebra_figures as cf
from _constants import (CLASS_NAMES, CLASS_COLORS, DISPLAY_ORDER, N_CLASSES,
                        OUTCOME_GOOD_RGB, OUTCOME_BAD_RGB)

FEATURE_STREAMS = ['cebra', 'protopnet', 'proto_acts', 'labelfreq', 'qeeg']

# Analogue trajectory colours. Every hue on the globe is already spoken for by
# one of the eight state regions, so the analogues are separated by VALUE, not
# hue: they keep the outcome semantics of the cohort cloud (blue = good,
# orange = poor) but sit far darker than the faint cloud and the thin state
# rings, which puts them unambiguously in the foreground.
ANALOGUE_GOOD = '#0b3d91'    # deep blue  (cloud good is light blue)
ANALOGUE_POOR = '#9c3400'    # deep rust  (cloud poor is light orange)


def confidence_profile(twin, pid, split='test'):
    """
    What the twin actually believed about a patient over time.
      p_final      P(good) at the last observed block
      p_mean       mean P(good) over observed blocks
      uncertainty  mean(1 - 2*|p - 0.5|); 1.0 = sat on the fence throughout
      crossings    number of times P(good) crossed 0.5
    """
    t4 = twin['step4']
    i = int(np.where(t4[f'{split}_pids'] == pid)[0][0])
    m = t4[f'mask_{split}'][i].astype(bool)
    p = t4[f'outcome_prob_{split}'][i][m]
    if len(p) == 0:
        return None
    return dict(pid=pid, index=i, n_obs=int(m.sum()),
                good=bool(t4[f'y_{split}'][i] == 1),
                cpc=int(t4[f'cpc_{split}'][i]),
                p_final=float(p[-1]), p_mean=float(p.mean()),
                uncertainty=float(np.mean(1.0 - 2.0 * np.abs(p - 0.5))),
                crossings=int(np.sum(np.diff(np.sign(p - 0.5)) != 0)),
                p=p)


def rank_by_cpc_confidence(twin, cpc_value, split='test', min_obs=40, top=8):
    """
    Exemplars at one exact CPC grade that the twin was most confident about,
    and right about.

    Confidence is scored toward the true label: P(good) for CPC 1-2,
    1 - P(good) for CPC 3-5. Ties broken by low mean uncertainty, so the
    trace is a clean commitment rather than a lucky endpoint.
    """
    t4 = twin['step4']
    rows = []
    for pid in t4[f'{split}_pids']:
        c = confidence_profile(twin, str(pid), split)
        if c is None or c['n_obs'] < min_obs or c['cpc'] != cpc_value:
            continue
        toward = c['p_final'] if c['good'] else 1.0 - c['p_final']
        mean_t = c['p_mean'] if c['good'] else 1.0 - c['p_mean']
        rows.append((str(pid), float(toward + mean_t - c['uncertainty']), c))
    rows.sort(key=lambda r: r[1], reverse=True)
    return rows[:top]


def rank_by_confidence(twin, archetype, split='test', min_obs=40, top=8):
    """
    Pick exemplars by what the MODEL believed, not by how the EEG states moved.
      'recovery'     good outcome the twin was confidently right about
      'nonrecovery'  poor outcome the twin was confidently right about
      'gray'         the twin never left the fence (highest uncertainty)
    """
    t4 = twin['step4']
    rows = []
    for pid in t4[f'{split}_pids']:
        c = confidence_profile(twin, str(pid), split)
        if c is None or c['n_obs'] < min_obs:
            continue
        if archetype == 'recovery':
            if not c['good']:
                continue
            score = c['p_final'] + c['p_mean'] - c['uncertainty']
        elif archetype == 'nonrecovery':
            if c['good']:
                continue
            score = (1 - c['p_final']) + (1 - c['p_mean']) - c['uncertainty']
        elif archetype == 'gray':
            score = c['uncertainty'] + 0.10 * min(c['crossings'], 6)
        else:
            raise ValueError(archetype)
        rows.append((str(pid), float(score), c))
    rows.sort(key=lambda r: r[1], reverse=True)
    return rows[:top]


def _guide(fig, x, y, row, col, color='gray', dash=None, width=1):
    """A reference line as a trace (add_vline/add_hline break on mixed 2D/3D)."""
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines',
                             line=dict(color=color, width=width, dash=dash),
                             showlegend=False, hoverinfo='skip'),
                  row=row, col=col)


def load_twin(twin_dir='twin'):
    return dict(
        step4=np.load(os.path.join(twin_dir, 'twin_step4_handoff.npz'),
                      allow_pickle=True),
        match=np.load(os.path.join(twin_dir, 'twin_matching_handoff.npz'),
                      allow_pickle=True),
    )


def _l2(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-12)


def retrieve_neighbors(twin, pid, hour=24, k=10, alpha=0.75):
    """Top-k training analogues for a test patient, as of `hour`."""
    t4, tm = twin['step4'], twin['match']
    cur = list(tm['current_hours'])
    if hour not in cur:
        raise ValueError(f'hour must be one of {cur}')
    hi = cur.index(hour)
    qi = int(np.where(tm['test_pids'] == pid)[0][0])

    # eligibility: train patients with observed EEG (no NaN) at this hour
    ok = ~np.isnan(tm['cebra_train'][:, hi, :]).any(axis=1)

    hq = t4['hidden_test'][qi, hour - 1]
    traj = _l2(t4['hidden_train'][:, hour - 1, :]) @ _l2(hq)

    feats = []
    for s in FEATURE_STREAMS:
        T = tm[f'{s}_train'][:, hi, :]
        q = tm[f'{s}_test'][qi, hi]
        mu = np.nanmean(T, axis=0)
        sd = np.nanstd(T, axis=0) + 1e-8
        feats.append(_l2(np.nan_to_num((T - mu) / sd)) @ _l2((q - mu) / sd))
    T, q = tm['clin_train'], tm['clin_test'][qi]
    mu, sd = T.mean(0), T.std(0) + 1e-8
    feats.append(_l2((T - mu) / sd) @ _l2((q - mu) / sd))
    feat = np.mean(feats, axis=0)

    comb = alpha * traj + (1 - alpha) * feat
    comb[~ok] = -np.inf
    top = np.argsort(-comb)[:k]
    return dict(
        query=pid, hour=hour, query_index=qi, alpha=alpha,
        eligible=int(ok.sum()),
        neighbors=[dict(pid=str(tm['train_pids'][j]), sim=float(comb[j]),
                        traj=float(traj[j]), feat=float(feat[j]),
                        cpc=int(t4['cpc_train'][j]), good=bool(t4['y_train'][j] == 1),
                        index=int(j)) for j in top],
    )


# Where the panel-b legend sits. The rolled-forward curves hug the top of the
# axis for a recovering patient and the bottom for a non-recovering one, so the
# legend has to move to the opposite corner or it covers them.
ROLL_LEGEND_POS = {
    'top-right':    dict(x=0.988, y=0.975, xanchor='right', yanchor='top'),
    'bottom-right': dict(x=0.988, y=0.545, xanchor='right', yanchor='bottom'),
    'top-left':     dict(x=0.600, y=0.975, xanchor='left',  yanchor='top'),
    'bottom-left':  dict(x=0.600, y=0.545, xanchor='left',  yanchor='bottom'),
}


def fig_twin_in_cebra_space(run, twin, pid, hour=24, k=10, alpha=0.75,
                            n_globe=3, split='train', cloud_subsample=35000,
                            roll_legend='bottom-right',
                            window=24, slerp_n=10, camera=None,
                            width=1450, height=760):
    """
    Figure 2 (CEBRA component) — the twin for one patient:
      (a) the query's observed trajectory on the manifold, with its retrieved
          analogues drawn in the same space and coloured by their outcome
      (b) the analogues' outcome trajectories rolled forward from the query
          hour, against the twin's own probability for the query
      (c) the query's EEG-state sequence, with the query hour marked
    """
    t4, tm = twin['step4'], twin['match']
    res = retrieve_neighbors(twin, pid, hour, k, alpha)
    qi = res['query_index']

    d = run[split]
    C = cf.class_centroids(run, split)
    camera = camera or cf.PAPER_CAMERA

    _, q = cf.find_patient(run, pid)
    obs = q['hours'] <= hour
    q_traj = cf.smooth_trajectory(q['emb'][obs], q['y'][obs], window, slerp_n)

    # Ten trajectories on one globe is unreadable; S2.9 makes the top-1 the
    # twin, so only the closest n_globe are drawn here. All k stay in panel b.
    n_traj = []
    for nb in res['neighbors'][:n_globe]:
        try:
            _, npat = cf.find_patient(run, nb['pid'])
        except KeyError:
            continue
        m = npat['hours'] <= hour
        if m.sum() < 2:
            continue
        n_traj.append((nb, cf.smooth_trajectory(npat['emb'][m], npat['y'][m],
                                                window, slerp_n)))

    paths = [q_traj['path']] + [t['path'] for _, t in n_traj]
    label_pts = [p[None, :] for p, _ in
                 cf.label_ring(run, camera, split).values()]
    rng = cf.axis_range(d['emb'], *paths, *label_pts, pad=1.04)

    fig = make_subplots(
        rows=2, cols=2,
        specs=[[{'type': 'scatter3d', 'rowspan': 2}, {'type': 'xy'}],
               [None, {'type': 'xy'}]],
        column_widths=[0.52, 0.48], row_heights=[0.62, 0.38],
        subplot_titles=(
            f'<b>a</b>   {pid} and its {len(n_traj)} closest analogues '
            f'in CEBRA space (as of hour {hour})',
            f'<b>b</b>   Outcome trajectories rolled forward from hour {hour}',
            f'<b>c</b>   {pid} EEG-state sequence'),
        horizontal_spacing=0.07, vertical_spacing=0.11)

    # ── (a) globe ────────────────────────────────────────────────────
    # here the cohort cloud is context, not the subject: keep it faint enough
    # that the analogue trajectories (same blue) still read against it
    cf._add_outcome_cloud(fig, d['emb'], d['cpc_b'], 1, 1, opacity=0.13,
                          size=1.4, legend=True, subsample=cloud_subsample,
                          legend_id='legend')
    cf._add_boundaries(fig, run, 1, 1, split=split, labels=True, legend=False,
                       camera=camera, width=1.8, label_size=11,
                       label_classes=cf.ATLAS_LABEL_CLASSES)
    seen = set()
    for rank, (nb, t) in enumerate(n_traj, 1):
        col = ANALOGUE_GOOD if nb['good'] else ANALOGUE_POOR
        top1 = rank == 1
        nm = (f"Twin (rank 1) — {nb['pid']}" if top1 else
              'Other analogues (rank 2-%d)' % len(n_traj))
        p = t['path']
        fig.add_trace(go.Scatter3d(          # halo, so the line reads on the cloud
            x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='lines',
            line=dict(color='rgba(255,255,255,0.85)', width=9 if top1 else 6),
            showlegend=False, hoverinfo='skip'), row=1, col=1)
        fig.add_trace(go.Scatter3d(
            x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='lines',
            line=dict(color=col, width=5.5 if top1 else 2.8),
            opacity=1.0 if top1 else 0.75,
            name=nm, legendgroup=nm, showlegend=nm not in seen,
            legend='legend',
            hovertemplate=f"{nb['pid']} rank {rank} (CPC {nb['cpc']})<extra></extra>"),
            row=1, col=1)
        seen.add(nm)
    cf._add_trajectory(fig, q_traj, 1, 1, name=f'{pid} (query)',
                       legend=True, legend_id='legend', line_width=7,
                       marker_size=3.4)

    # ── (b) rolled-forward outcome trajectories ──────────────────────
    cur = list(tm['current_hours'])
    fut = np.array(tm['future_hours'], float)
    hi = cur.index(hour)
    # show the same exemplars as the globe, not all k -- the rest are summarised
    # by the dotted aggregate below
    for rank, nb in enumerate(res['neighbors'][:n_globe], 1):
        col = ANALOGUE_GOOD if nb['good'] else ANALOGUE_POOR
        top1 = rank == 1
        fig.add_trace(go.Scatter(
            x=fut, y=tm['roll_traj_train'][nb['index'], hi, :],
            mode='lines',
            line=dict(color=col, width=3.0 if top1 else 1.6),
            opacity=1.0 if top1 else 0.7,
            name=(f"Twin (rank 1) — {nb['pid']}" if top1 else
                  f'Other analogues (rank 2-{n_globe})'),
            legendgroup=('twin1' if top1 else 'twinrest'),
            showlegend=(top1 or rank == 2), legend='legend2',
            hovertemplate=f"{nb['pid']} rank {rank}: %{{y:.2f}}<extra></extra>"),
            row=1, col=2)
    fwd = fut >= hour
    nb_mean = np.nanmean(
        [tm['roll_traj_train'][nb['index'], hi, :][fwd]
         for nb in res['neighbors']], axis=0)
    fig.add_trace(go.Scatter(x=fut[fwd], y=nb_mean, mode='lines',
                             line=dict(color='#444', width=2.2, dash='dot'),
                             name=f'Mean of all {len(res["neighbors"])} analogues',
                             legend='legend2'),
                  row=1, col=2)
    mk = t4['mask_test'][qi].astype(bool)
    hrs = np.arange(1, 85)
    fig.add_trace(go.Scatter(x=hrs[mk], y=t4['outcome_prob_test'][qi][mk],
                             mode='lines', line=dict(color='#111', width=3),
                             name=f'{pid} twin P(good)', legend='legend2'),
                  row=1, col=2)
    # guide lines drawn as traces: add_vline/add_hline choke on scatter3d
    # traces living in the same figure (they probe trace['xaxis'])
    _guide(fig, [hour, hour], [0, 1], 1, 2, dash='dash', color='gray')
    _guide(fig, [0, 85], [0.5, 0.5], 1, 2, color='lightgray')

    # ── (c) state sequence ───────────────────────────────────────────
    for c in DISPLAY_ORDER:
        m = q['y'] == c
        if not m.any():
            continue
        fig.add_trace(go.Scatter(
            x=q['hours'][m],
            y=np.full(m.sum(), DISPLAY_ORDER.index(c)),
            mode='markers',
            marker=dict(color=CLASS_COLORS[c], size=4, symbol='line-ns',
                        line=dict(color=CLASS_COLORS[c], width=3)),
            showlegend=False, name=CLASS_NAMES[c],
            hovertemplate=f'{CLASS_NAMES[c]} @ %{{x:.1f}} h<extra></extra>'),
            row=2, col=2)
    _guide(fig, [hour, hour], [-0.6, N_CLASSES - 0.4], 2, 2,
           dash='dash', color='gray')

    fig.update_xaxes(title_text='Hours from recording start', row=1, col=2,
                     range=[0, 85])
    fig.update_yaxes(title_text='P(good outcome)', row=1, col=2, range=[0, 1])
    fig.update_xaxes(title_text='Hours from recording start', row=2, col=2,
                     range=[0, 85])
    fig.update_yaxes(row=2, col=2, tickmode='array',
                     tickvals=list(range(N_CLASSES)),
                     ticktext=[CLASS_NAMES[c] for c in DISPLAY_ORDER],
                     autorange='reversed', tickfont=dict(size=9))

    gf = float(np.mean([n['good'] for n in res['neighbors']]))
    fig.update_layout(
        scene=cf._scene(rng, camera),
        paper_bgcolor='white', plot_bgcolor='white',
        font=dict(color='black', family=cf.FONT, size=12),
        legend=dict(title=dict(text='<b>CEBRA space</b>', font=dict(size=10)),
                    font=dict(size=9.5), itemsizing='constant',
                    x=0.005, y=0.33, xanchor='left', yanchor='top',
                    bgcolor='rgba(255,255,255,0.85)',
                    bordercolor='lightgray', borderwidth=1),
        legend2=dict(font=dict(size=9.5), itemsizing='constant',
                     bgcolor='rgba(255,255,255,0.88)',
                     bordercolor='lightgray', borderwidth=1,
                     **ROLL_LEGEND_POS[roll_legend]),
        margin=dict(l=0, r=20, t=76, b=50),
        width=width, height=height,
        annotations=list(fig.layout.annotations) + [dict(
            text=(f"analogue good-fraction {gf:.0%}  ·  "
                  f"twin P(good) at h{hour} = "
                  f"{t4['outcome_prob_test'][qi, hour - 1]:.2f}  ·  "
                  f"true outcome CPC {t4['cpc_test'][qi]}"),
            xref='paper', yref='paper', x=0.76, y=1.075, showarrow=False,
            font=dict(size=10.5, color='#444'), xanchor='center')])
    for a in fig.layout.annotations[:3]:
        a.font.size = 12.5
        a.font.family = cf.FONT
    fig.update_xaxes(showgrid=True, gridcolor='#eee', zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor='#eee', zeroline=False)
    return fig


In [ ]:
%%writefile 09_fig3_trajectory_globes.py
"""
09_fig3_trajectory_globes.py — Figure 3, CEBRA trajectory globes.

Atlas + one globe per outcome grade (CPC 1 / 3 / 5), each the patient the twin
called most confidently and correctly at that grade -- so the panels differ by
what the MODEL believed, not by how the EEG states happened to move.

All four globes share one camera. In the interactive HTML they rotate together:
drag any globe and the other three follow, and the overlay prints the camera as
a dict you can paste into CAMERA below to lock the view for the static export.
"""
import os
import numpy as np
import cebra_figures as cf
import twin_figures as tf

RUN_TAG   = 'combo_v'
SPLIT     = 'train'          # split providing the manifold/atlas cloud
MIN_OBS   = 40               # observed 1-h blocks required of a candidate
WINDOW    = 24
SLERP_N   = 10
STEM      = 'Fig3_cebra_globes'
SINGLE_GLOBE_HTML = True
CAMERA    = dict(eye=dict(x=0.706, y=-0.617, z=0.612),
               up=dict(x=-0.4, y=0.372, z=0.838))
TWIN_DIR  = os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', 'twin')

# (pid or None, caption, CPC grade). None -> most confidently, correctly called
# patient at that grade.
PANELS = [
    (None, 'Recovery',     1),
    (None, 'Intermediate', 3),
    (None, 'Non-recovery', 5),
]

if __name__ == '__main__':
    run  = cf.load_run(RUN_TAG)
    twin = tf.load_twin(TWIN_DIR)
    print(f"Loaded {RUN_TAG}: train {run['train']['emb'].shape}, "
          f"test {run['test']['emb'].shape}")

    panels, chosen = [], []
    for pid, caption, cpc in PANELS:
        ranked = [r for r in tf.rank_by_cpc_confidence(twin, cpc, 'test', MIN_OBS)
                  if r[0] not in chosen]
        print(f'\nCPC {cpc} — most confidently, correctly called:')
        for q, score, c in ranked[:4]:
            print(f"  {q}  score={score:+.3f}  P_final={c['p_final']:.2f}  "
                  f"P_mean={c['p_mean']:.2f}  uncert={c['uncertainty']:.2f}")
        pid = pid or ranked[0][0]
        chosen.append(pid)
        c = tf.confidence_profile(twin, pid)
        print(f"  -> {caption} (CPC {cpc}): {pid}  P_final={c['p_final']:.2f}")
        panels.append((pid, f'{caption} (CPC {cpc})'))

    fig = cf.fig_trajectory_globes(run, patients=panels, split=SPLIT,
                                   window=WINDOW, slerp_n=SLERP_N,
                                   camera=CAMERA)
    cf.export(fig, STEM)

    if SINGLE_GLOBE_HTML:
        print('\nStandalone globes:')
        cf.export_single_globes(run, panels, split=SPLIT, window=WINDOW,
                                slerp_n=SLERP_N, camera=CAMERA)
    print('\nDone.')


In [ ]:
%%writefile 10_fig2_twin_in_cebra_space.py
"""
10_fig2_twin_in_cebra_space.py — Figure 2 (CEBRA component of the digital twin).

Two panels of the paper figure: one recovering and one non-recovering patient,
each shown with the analogues the twin actually retrieved at HOUR.

Both were picked to have a COMPLEX course -- all eight ACNS states visited, many
transitions -- so the figure demonstrates that the embedding and the retrieval
hold up on a hard trajectory, not just a clean one. Both are also patients the
twin called confidently and correctly.

Patient-agnostic: PATIENTS takes any test-split IDs, HOUR any value in
current_hours (6, 12, ... 84).
"""
import os
import cebra_figures as cf
import twin_figures as tf

RUN_TAG  = 'combo_v'
HOUR     = 24          # bedside "as of" hour; must be in current_hours
TOP_K    = 10          # neighbours retrieved (globe shows the closest N_GLOBE)
N_GLOBE  = 3
ALPHA    = 0.75        # manuscript S2.8
# Per-patient camera and panel-b legend corner. The rolled-forward curves sit
# high for a recovering patient and low for a non-recovering one, so the legend
# goes to the opposite corner in each case.
CAM_GOOD = dict(eye=dict(x=-0.156, y=1.826, z=-0.802),
                up=dict(x=-0.275, y=0.367, z=0.889))
CAM_POOR = dict(eye=dict(x=0.827, y=-0.879, z=0.695),
                up=dict(x=0.204, y=0.718, z=0.665))
TWIN_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', 'twin')

# (patient, role). Both traverse all 8 states; see rank_by_complexity below.
PATIENTS = [
    ('ICARE_0010', 'recovering',     CAM_GOOD, 'bottom-right'),
    ('ICARE_0077', 'non-recovering', CAM_POOR, 'top-right'),
]

if __name__ == '__main__':
    run  = cf.load_run(RUN_TAG)
    twin = tf.load_twin(TWIN_DIR)

    for pid, role, camera, roll_legend in PATIENTS:
        c = tf.confidence_profile(twin, pid)
        res = tf.retrieve_neighbors(twin, pid, HOUR, TOP_K, ALPHA)
        gf = sum(n['good'] for n in res['neighbors']) / len(res['neighbors'])
        print(f"\n{pid} ({role})  CPC={c['cpc']}  P_final={c['p_final']:.2f}  "
              f"P(good)@h{HOUR}={c['p'][min(HOUR, len(c['p'])) - 1]:.2f}")
        print(f"  pool {res['eligible']}/695 · analogue good-fraction {gf:.0%}")
        for i, nb in enumerate(res['neighbors'][:N_GLOBE], 1):
            print(f"   {i}. {nb['pid']}  sim={nb['sim']:.3f}  CPC={nb['cpc']}  "
                  f"{'GOOD' if nb['good'] else 'poor'}")

        fig = tf.fig_twin_in_cebra_space(run, twin, pid, HOUR, TOP_K, ALPHA,
                                         n_globe=N_GLOBE, camera=camera,
                                         roll_legend=roll_legend)
        cf.export(fig, f'Fig2_twin_cebra_{pid}_h{HOUR}')
    print('\nDone.')


In [ ]:
%%writefile 11_fig_prototypes.py
"""
11_fig_prototypes.py — CEBRA manifold + ProtoPNet prototypes for one patient.

Patient-agnostic: set PID to anyone in either split.
"""
import numpy as np
import cebra_figures as cf
from _constants import CLASS_NAMES

RUN_TAG     = 'combo_v'
PID         = 'ICARE_0279'
SPLIT       = 'train'      # manifold cloud
TOP_K       = 50           # segments averaged to place each prototype
N_HIGHLIGHT = 5
WINDOW      = 24
STEM        = f'Fig_prototypes_{PID}'

if __name__ == '__main__':
    run = cf.load_run(RUN_TAG)
    pos, pstate = cf.prototype_landmarks(run, SPLIT, TOP_K)
    counts = {CLASS_NAMES[c]: int((pstate == c).sum()) for c in np.unique(pstate)}
    print(f'{len(pos)} prototypes by phenotype: {counts}')
    missing = [CLASS_NAMES[c] for c in range(8) if c not in np.unique(pstate)]
    if missing:
        print(f'  no prototype codes for: {", ".join(missing)}')

    _, p = cf.find_patient(run, PID)
    src = run['test'] if PID in run['test']['pid'] else run['train']
    m = src['pid'] == PID
    mean_act = src['acts'][m].mean(axis=0)
    print(f'\n{PID} top-{N_HIGHLIGHT} prototypes by mean activation:')
    for i in np.argsort(-mean_act)[:N_HIGHLIGHT]:
        print(f'  prototype {i:2d}  {CLASS_NAMES[pstate[i]]:14s} '
              f'mean={mean_act[i]:.3f}')

    fig = cf.fig_patient_prototypes(run, PID, split=SPLIT, top_k=TOP_K,
                                    window=WINDOW, n_highlight=N_HIGHLIGHT)
    cf.export(fig, STEM)

    # globe alone, square and full-size: rotate freely, the overlay prints the
    # camera as a dict you can paste back as camera=...
    globe = cf.fig_patient_prototypes(run, PID, split=SPLIT, top_k=TOP_K,
                                      window=WINDOW, n_highlight=N_HIGHLIGHT,
                                      panel_b=False)
    cf.export(globe, f'{STEM}_globe', formats=('html',))
    print('\nDone.')


## 4. Step 01 — preprocessing (`combo_v`)

In [ ]:
import re, os, sys
sys.path.insert(0, os.getcwd())
src = open('01_data_preprocessing.py').read()
ov = {'RUN_TAG':'combo_v', 'FEATURE_KEYS':['features','cebra_features'],
      'PCA_KEY':'features', 'PCA_COMPONENTS':50, 'SEED':42, 'BIN_SEC':300}
for k, v in ov.items():
    src, n = re.subn(rf'(?m)^{re.escape(k)}\s*=.*$', f'{k} = {v!r}', src, count=1); assert n == 1, k
exec(compile(src, '01_data_preprocessing.py', 'exec'),
     {'__name__':'__main__', '__file__':'01_data_preprocessing.py'})


## 5. Step 02 — CEBRA hybrid training (GPU)

Appendix E settings: 3-D output, 16 units, tau = 0.5, time offset 144 bins (12 h),
batch 1024, lr 3e-4, 20 000 iterations, seed 42.
`combo_v` = time + `predictions` + `cpc_binary` + `probabilities`. ~3-6 min on a T4.

In [ ]:
import re
src = open('02_cebra_hybrid_training.py').read()
ov = {'RUN_TAG':'combo_v', 'LABEL_KEYS_DISC':['predictions','cpc_binary'],
      'LABEL_KEYS_CONT':['probabilities'], 'USE_TIME_OBJECTIVE':True,
      'OUTPUT_DIM':3, 'TIME_OFFSET':144, 'BATCH_SIZE':1024, 'MAX_ITER':20000,
      'TEMPERATURE':0.5, 'NUM_UNITS':16, 'LR':3e-4, 'KNN_NEIGHBORS':10, 'SEED':42}
for k, v in ov.items():
    src, n = re.subn(rf'(?m)^{re.escape(k)}\s*=.*$', f'{k} = {v!r}', src, count=1); assert n == 1, k
exec(compile(src, '02_cebra_hybrid_training.py', 'exec'),
     {'__name__':'__main__', '__file__':'02_cebra_hybrid_training.py'})


## 6. Figure 3 — CEBRA trajectory globes

Four globes: state atlas, then one panel per archetype the abstract claims
("distinct recovery, non-recovery, and intermediate trajectories").
Edit `PANELS` for different patients; `None` auto-selects.

In [ ]:
import importlib, cebra_figures as cf
importlib.reload(cf)
run = cf.load_run('combo_v')
C   = cf.class_centroids(run)

for outcome in ('good', 'poor'):
    print(f'\nTop {outcome}-outcome candidates by recovery-axis arc:')
    for pid, score, s in cf.rank_by_arc(run, 'test', outcome, 40, C)[:5]:
        print(f"  {pid}  score={score:+.3f}  CPC={s['cpc']:.0f}  "
              f"{s['hours']:5.1f}h  ->cont={s['to_continuous']:+.3f}")


In [ ]:
import twin_figures as tf; importlib.reload(tf)
twin = tf.load_twin('/content/twin')

panels = []
for cpc, caption in [(1, 'Recovery'), (3, 'Intermediate'), (5, 'Non-recovery')]:
    ranked = [r for r in tf.rank_by_cpc_confidence(twin, cpc, 'test', 40)
              if r[0] not in [p[0] for p in panels]]
    print(f'\nCPC {cpc} — most confidently, correctly called:')
    for q, sc, c_ in ranked[:4]:
        print(f"  {q}  score={sc:+.3f}  P_final={c_['p_final']:.2f}  "
              f"uncert={c_['uncertainty']:.2f}")
    panels.append((ranked[0][0], f'{caption} (CPC {cpc})'))
print('\npanels:', panels)

CAMERA = dict(eye=dict(x=0.706, y=-0.617, z=0.612),
              up=dict(x=-0.4, y=0.372, z=0.838))
fig3 = cf.fig_trajectory_globes(run, patients=panels, window=24, camera=CAMERA)
cf.export(fig3, 'Fig3_cebra_globes')

# standalone globes; in the 4-panel HTML all scenes rotate together
cf.export_single_globes(run, panels, window=24, camera=CAMERA)
fig3.show()


## 7. Figure 2 — the digital twin in CEBRA space

Retrieval reproduces the deployed rule (S2.8):
`combined = 0.75 * trajectory + 0.25 * feature`, restricted to training
patients with EEG observed by the query hour.

In [ ]:
if not HAVE_TWIN:
    print('twin handoff files missing - skipping Figure 2')
else:
    import twin_figures as tf; importlib.reload(tf)
    twin = tf.load_twin('/content/twin')
    HOUR, TOP_K, N_GLOBE, ALPHA = 24, 10, 3, 0.75

    CAM_GOOD = dict(eye=dict(x=-0.156, y=1.826, z=-0.802),
                    up=dict(x=-0.275, y=0.367, z=0.889))
    CAM_POOR = dict(eye=dict(x=0.827, y=-0.879, z=0.695),
                    up=dict(x=0.204, y=0.718, z=0.665))

    # one recovering, one non-recovering; both traverse all eight ACNS states.
    # The panel-b legend goes to whichever corner the rolled-forward curves
    # leave free: they run high for the recovering patient, low for the other.
    JOBS = [('ICARE_0010', 'recovering',     CAM_GOOD, 'bottom-right'),
            ('ICARE_0077', 'non-recovering', CAM_POOR, 'top-right')]

    for PID, role, camera, roll_legend in JOBS:
        c_ = tf.confidence_profile(twin, PID)
        res = tf.retrieve_neighbors(twin, PID, HOUR, TOP_K, ALPHA)
        gf = sum(n['good'] for n in res['neighbors']) / len(res['neighbors'])
        print(f"\n{PID} ({role})  CPC={c_['cpc']}  P_final={c_['p_final']:.2f}  "
              f"analogue good-fraction {gf:.0%}")
        for i, nb_ in enumerate(res['neighbors'][:N_GLOBE], 1):
            print(f"   {i}. {nb_['pid']}  sim={nb_['sim']:.3f}  CPC={nb_['cpc']}")
        fig2 = tf.fig_twin_in_cebra_space(run, twin, PID, HOUR, TOP_K, ALPHA,
                                          n_globe=N_GLOBE, camera=camera,
                                          roll_legend=roll_legend)
        cf.export(fig2, f'Fig2_twin_cebra_{PID}_h{HOUR}')
        fig2.show()


## 8. Prototype figure — patient + ProtoPNet landmarks in CEBRA space

Each of the 45 prototypes is placed at the mean embedding of its top-50
activating segments and coloured by the phenotype it codes for. Panel b is the
patient's activation trail behind the trajectory in panel a.

In [ ]:
import numpy as np
PID_PROTO = 'ICARE_0279'

pos, pstate = cf.prototype_landmarks(run, 'train', 50)
from _constants import CLASS_NAMES
print({CLASS_NAMES[c]: int((pstate == c).sum()) for c in np.unique(pstate)})
missing = [CLASS_NAMES[c] for c in range(8) if c not in np.unique(pstate)]
if missing:
    print('no prototype codes for:', ', '.join(missing))

figp = cf.fig_patient_prototypes(run, PID_PROTO, split='train', top_k=50,
                                 window=24, n_highlight=5)
cf.export(figp, f'Fig_prototypes_{PID_PROTO}')
figp.show()


## 8. Save everything back to Drive

In [ ]:
import shutil, os, glob
DEST = '/content/drive/MyDrive/EEG_Twin_CEBRA_outputs'
os.makedirs(DEST, exist_ok=True)
for pat in ('cebra_embeddings_*', 'cebra_hybrid_model_*', 'cebra_pca_scaler_*'):
    for f in glob.glob(f"{os.environ['CEBRA_OUT_DIR']}/{pat}"):
        shutil.copy2(f, DEST)
figdir = os.path.join(DEST, 'figures'); os.makedirs(figdir, exist_ok=True)
for f in glob.glob(f"{os.environ['CEBRA_OUT_DIR']}/figures/*"):
    shutil.copy2(f, figdir)
print('saved to', DEST)
for f in sorted(glob.glob(DEST + '/**/*', recursive=True)):
    print(f'   {f.replace(DEST + "/", ""):46s} {os.path.getsize(f)/1e6:6.1f} MB')
